# Module 4 — Text Representation & Vector Space Model
**Demo • Guided Practice • Exercises**

Nội dung chính:
- Vocabulary (từ vựng)
- Bag-of-Words (BoW)
- Count vectors
- Document–Term Matrix (DTM)
- TF–IDF
- Cosine similarity
- Sparse representation
- Out-of-vocabulary (OOV)

> Mục tiêu: biến văn bản thành vector số để máy học có thể tính toán, so sánh và đưa vào các mô hình ML.

## Learning outcomes

Sau buổi thực hành, sinh viên có thể:

1. Xây dựng vocabulary từ một corpus.
2. Biểu diễn tài liệu bằng BoW/count vectors.
3. Tạo và đọc Document–Term Matrix.
4. Giải thích và tính TF–IDF.
5. Tính cosine similarity giữa các tài liệu / giữa query và document.
6. Giải thích vì sao ma trận văn bản thường là sparse.
7. Nhận biết và xử lý cơ bản vấn đề OOV.
8. Sử dụng `CountVectorizer` và `TfidfVectorizer` của scikit-learn.

In [ ]:
# Nếu chạy trên Google Colab, các thư viện sau thường đã có sẵn.
# Nếu cần, bỏ dấu # ở dòng dưới:
# !pip install -q scikit-learn pandas scipy matplotlib


import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 50)

## Corpus dùng xuyên suốt notebook

Ta dùng một corpus nhỏ để có thể kiểm tra kết quả bằng tay.

In [2]:
docs = [
    "machine learning learns from data",
    "natural language processing works with text data",
    "machine learning can classify text",
    "language models learn patterns from text",
]

for i, d in enumerate(docs, start=1):
    print(f"D{i}: {d}")

D1: machine learning learns from data
D2: natural language processing works with text data
D3: machine learning can classify text
D4: language models learn patterns from text


# Part A — Demo

## Demo 1. Vocabulary

**Vocabulary** là tập các token duy nhất mà hệ thống dùng làm các chiều của vector.

Với mô hình BoW cổ điển:
- mỗi từ trong vocabulary tương ứng với một chiều;
- thứ tự từ trong câu thường bị bỏ qua;
- giá trị của chiều có thể là `0/1`, số lần xuất hiện, hoặc TF–IDF.

In [3]:
def simple_tokenize(text):
    # tokenizer minh họa: lowercase + lấy chuỗi chữ/số
    return re.findall(r"\b\w+\b", text.lower())

tokenized_docs = [simple_tokenize(d) for d in docs]
vocab = sorted(set(token for doc in tokenized_docs for token in doc))

print("Vocabulary size:", len(vocab))
print(vocab)

Vocabulary size: 16
['can', 'classify', 'data', 'from', 'language', 'learn', 'learning', 'learns', 'machine', 'models', 'natural', 'patterns', 'processing', 'text', 'with', 'works']


### Ánh xạ token → index

In [4]:
token_to_id = {token: i for i, token in enumerate(vocab)}
token_to_id

{'can': 0,
 'classify': 1,
 'data': 2,
 'from': 3,
 'language': 4,
 'learn': 5,
 'learning': 6,
 'learns': 7,
 'machine': 8,
 'models': 9,
 'natural': 10,
 'patterns': 11,
 'processing': 12,
 'text': 13,
 'with': 14,
 'works': 15}

## Demo 2. Bag-of-Words và count vector — làm bằng tay

In [5]:
def bow_vector(tokens, vocabulary):
    counts = Counter(tokens)
    return np.array([counts[word] for word in vocabulary], dtype=int)

manual_vectors = np.vstack([
    bow_vector(tokens, vocab) for tokens in tokenized_docs
])

manual_dtm = pd.DataFrame(
    manual_vectors,
    index=[f"D{i}" for i in range(1, len(docs)+1)],
    columns=vocab
)

manual_dtm

,can,classify,data,from,language,learn,learning,learns,machine,models,natural,patterns,processing,text,with,works
D1,0,0,1,1,0,0,1,1,1,0,0,0,0,0,0,0
D2,0,0,1,0,1,0,0,0,0,0,1,0,1,1,1,1
D3,1,1,0,0,0,0,1,0,1,0,0,0,0,1,0,0
D4,0,0,0,1,1,1,0,0,0,1,0,1,0,1,0,0


**Cách đọc:** nếu ô `(D3, text) = 1`, nghĩa là từ `text` xuất hiện 1 lần trong document D3.

Đây chính là **Document–Term Matrix (DTM)**:
- hàng = document;
- cột = term/token;
- giá trị = trọng số của term trong document.

## Demo 3. `CountVectorizer`

In [6]:
count_vec = CountVectorizer()
X_count = count_vec.fit_transform(docs)

count_df = pd.DataFrame(
    X_count.toarray(),
    index=[f"D{i}" for i in range(1, len(docs)+1)],
    columns=count_vec.get_feature_names_out()
)

print("Shape:", X_count.shape)
print("Type :", type(X_count))
count_df

Shape: (4, 16)
Type : <class 'scipy.sparse._csr.csr_matrix'>


,can,classify,data,from,language,learn,learning,learns,machine,models,natural,patterns,processing,text,with,works
D1,0,0,1,1,0,0,1,1,1,0,0,0,0,0,0,0
D2,0,0,1,0,1,0,0,0,0,0,1,0,1,1,1,1
D3,1,1,0,0,0,0,1,0,1,0,0,0,0,1,0,0
D4,0,0,0,1,1,1,0,0,0,1,0,1,0,1,0,0


### Binary BoW vs Count BoW

In [7]:
binary_vec = CountVectorizer(binary=True)
X_binary = binary_vec.fit_transform(docs)

pd.DataFrame(
    X_binary.toarray(),
    index=[f"D{i}" for i in range(1, len(docs)+1)],
    columns=binary_vec.get_feature_names_out()
)

,can,classify,data,from,language,learn,learning,learns,machine,models,natural,patterns,processing,text,with,works
D1,0,0,1,1,0,0,1,1,1,0,0,0,0,0,0,0
D2,0,0,1,0,1,0,0,0,0,0,1,0,1,1,1,1
D3,1,1,0,0,0,0,1,0,1,0,0,0,0,1,0,0
D4,0,0,0,1,1,1,0,0,0,1,0,1,0,1,0,0


## Demo 4. TF–IDF

Một dạng phổ biến:



\[
tfidf(t,d) = tf(t,d) \times idf(t)
\]

với

\[
idf(t) = \log\left(\frac{N}{df(t)}\right) + 1
\]

Trong đó:
- `tf(t,d)`: số lần term `t` xuất hiện trong document `d`;
- `df(t)`: số document có chứa `t`;
- `N`: tổng số document.

Ý tưởng:
- từ xuất hiện nhiều trong **một** document → quan trọng hơn;
- từ xuất hiện ở **quá nhiều** document → ít khả năng phân biệt.

In [8]:
# Tính thủ công TF-IDF theo công thức phía trên
N = len(docs)

term_df = {}
for term in vocab:
    term_df[term] = sum(term in tokens for tokens in tokenized_docs)

idf_manual = {
    term: math.log(N / term_df[term]) + 1
    for term in vocab
}

tfidf_manual = manual_dtm.astype(float).copy()
for term in vocab:
    tfidf_manual[term] = tfidf_manual[term] * idf_manual[term]

print("IDF:")
display(pd.Series(idf_manual).sort_values())
print("\nTF-IDF (manual):")
display(tfidf_manual.round(3))

IDF:


text          1.287682
from          1.693147
learning      1.693147
data          1.693147
machine       1.693147
language      1.693147
can           2.386294
classify      2.386294
learns        2.386294
learn         2.386294
natural       2.386294
models        2.386294
patterns      2.386294
processing    2.386294
with          2.386294
works         2.386294
dtype: float64


TF-IDF (manual):


,can,classify,data,from,language,learn,learning,learns,machine,models,natural,patterns,processing,text,with,works
D1,0.000,0.000,1.693,1.693,0.000,0.000,1.693,2.386,1.693,0.000,0.000,0.000,0.000,0.000,0.000,0.000
D2,0.000,0.000,1.693,0.000,1.693,0.000,0.000,0.000,0.000,0.000,2.386,0.000,2.386,1.288,2.386,2.386
D3,2.386,2.386,0.000,0.000,0.000,0.000,1.693,0.000,1.693,0.000,0.000,0.000,0.000,1.288,0.000,0.000
D4,0.000,0.000,0.000,1.693,1.693,2.386,0.000,0.000,0.000,2.386,0.000,2.386,0.000,1.288,0.000,0.000


### So sánh với `TfidfVectorizer`

Để khớp với công thức thủ công ở trên, ta tắt smoothing và normalization.

In [9]:
tfidf_exact = TfidfVectorizer(
    smooth_idf=False,
    norm=None,
    use_idf=True,
    sublinear_tf=False
)
X_tfidf_exact = tfidf_exact.fit_transform(docs)

tfidf_exact_df = pd.DataFrame(
    X_tfidf_exact.toarray(),
    index=[f"D{i}" for i in range(1, len(docs)+1)],
    columns=tfidf_exact.get_feature_names_out()
)

tfidf_exact_df.round(3)

,can,classify,data,from,language,learn,learning,learns,machine,models,natural,patterns,processing,text,with,works
D1,0.000,0.000,1.693,1.693,0.000,0.000,1.693,2.386,1.693,0.000,0.000,0.000,0.000,0.000,0.000,0.000
D2,0.000,0.000,1.693,0.000,1.693,0.000,0.000,0.000,0.000,0.000,2.386,0.000,2.386,1.288,2.386,2.386
D3,2.386,2.386,0.000,0.000,0.000,0.000,1.693,0.000,1.693,0.000,0.000,0.000,0.000,1.288,0.000,0.000
D4,0.000,0.000,0.000,1.693,1.693,2.386,0.000,0.000,0.000,2.386,0.000,2.386,0.000,1.288,0.000,0.000


In [10]:
# Kiểm tra 2 cách tính có bằng nhau không
cols = tfidf_exact.get_feature_names_out().tolist()
manual_aligned = tfidf_manual[cols].to_numpy()

print("Max absolute difference:",
      np.abs(manual_aligned - X_tfidf_exact.toarray()).max())

Max absolute difference: 0.0


### TF–IDF mặc định của scikit-learn

`TfidfVectorizer()` mặc định còn:
- dùng smoothed IDF;
- chuẩn hóa mỗi vector theo L2 norm.

In [11]:
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(docs)

tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    index=[f"D{i}" for i in range(1, len(docs)+1)],
    columns=tfidf_vec.get_feature_names_out()
)

tfidf_df.round(3)

,can,classify,data,from,language,learn,learning,learns,machine,models,natural,patterns,processing,text,with,works
D1,0.000,0.000,0.422,0.422,0.000,0.000,0.422,0.536,0.422,0.000,0.000,0.000,0.000,0.000,0.000,0.000
D2,0.000,0.000,0.332,0.000,0.332,0.000,0.000,0.000,0.000,0.000,0.421,0.000,0.421,0.269,0.421,0.421
D3,0.523,0.523,0.000,0.000,0.000,0.000,0.413,0.000,0.413,0.000,0.000,0.000,0.000,0.334,0.000,0.000
D4,0.000,0.000,0.000,0.366,0.366,0.464,0.000,0.000,0.000,0.464,0.000,0.464,0.000,0.296,0.000,0.000


## Demo 5. Cosine similarity

Với hai vector \(x\) và \(y\):



\[
\cos(x,y) = \frac{x \cdot y}{\|x\|\|y\|}
\]

- gần `1`: hướng vector gần nhau → nội dung biểu diễn tương tự;
- gần `0`: ít đặc trưng chung;
- với BoW/TF-IDF không âm, cosine thường nằm trong `[0,1]`.

In [12]:
sim_matrix = cosine_similarity(X_tfidf)

sim_df = pd.DataFrame(
    sim_matrix,
    index=[f"D{i}" for i in range(1, len(docs)+1)],
    columns=[f"D{i}" for i in range(1, len(docs)+1)]
)

sim_df.round(3)

,D1,D2,D3,D4
D1,1.000,0.140,0.348,0.154
D2,0.140,1.000,0.090,0.201
D3,0.348,0.090,1.000,0.099
D4,0.154,0.201,0.099,1.000


### Mini information retrieval: query → documents

In [13]:
query = "machine learning for text data"

q_vec = tfidf_vec.transform([query])
scores = cosine_similarity(q_vec, X_tfidf).ravel()

ranking = pd.DataFrame({
    "document": [f"D{i}" for i in range(1, len(docs)+1)],
    "text": docs,
    "cosine_similarity": scores
}).sort_values("cosine_similarity", ascending=False)

ranking

,document,text,cosine_similarity
0,D1,machine learning learns from data,0.662549
2,D3,machine learning can classify text,0.573108
1,D2,natural language processing works with text data,0.287175
3,D4,language models learn patterns from text,0.125330


## Demo 6. Sparse representation

Trong corpus lớn:
- vocabulary có thể có hàng chục nghìn hoặc hàng trăm nghìn term;
- mỗi document chỉ chứa một phần rất nhỏ vocabulary;
- vì vậy DTM chứa rất nhiều số `0`.

Lưu toàn bộ số 0 là lãng phí → dùng **sparse matrix**.

In [14]:
print("Matrix shape       :", X_count.shape)
print("Stored non-zeros   :", X_count.nnz)
total_cells = X_count.shape[0] * X_count.shape[1]
density = X_count.nnz / total_cells

print("Total cells        :", total_cells)
print("Density            :", round(density, 4))
print("Sparsity           :", round(1 - density, 4))
print("Sparse matrix type :", type(X_count))

Matrix shape       : (4, 16)
Stored non-zeros   : 23
Total cells        : 64
Density            : 0.3594
Sparsity           : 0.6406
Sparse matrix type : <class 'scipy.sparse._csr.csr_matrix'>


In [15]:
# Ví dụ corpus lớn giả lập để thấy mức độ sparse rõ hơn
rng = np.random.default_rng(42)

n_docs = 5000
vocab_size = 20000
nnz_per_doc = 50

rows = np.repeat(np.arange(n_docs), nnz_per_doc)
cols = np.concatenate([
    rng.choice(vocab_size, size=nnz_per_doc, replace=False)
    for _ in range(n_docs)
])
data = np.ones(len(rows), dtype=np.float32)

X_sparse_demo = csr_matrix(
    (data, (rows, cols)),
    shape=(n_docs, vocab_size)
)

dense_bytes = n_docs * vocab_size * 4  # float32

sparse_bytes = (
    X_sparse_demo.data.nbytes
    + X_sparse_demo.indices.nbytes
    + X_sparse_demo.indptr.nbytes
)

print(f"Dense estimate : {dense_bytes / 1024**2:.1f} MB")
print(f"Sparse CSR     : {sparse_bytes / 1024**2:.1f} MB")
print(f"Density        : {X_sparse_demo.nnz / (n_docs*vocab_size):.6f}")

Dense estimate : 381.5 MB
Sparse CSR     : 1.9 MB
Density        : 0.002500


## Demo 7. OOV — Out-of-Vocabulary

OOV là token không có trong vocabulary đã học từ training corpus.

Với `CountVectorizer` / `TfidfVectorizer`:
- `fit(...)`: xây vocabulary từ dữ liệu huấn luyện;
- `transform(...)`: chỉ dùng vocabulary đã có;
- token mới/OOV thường bị bỏ qua.

In [16]:
train_docs = [
    "cat eats fish",
    "dog eats meat",
    "cat and dog are pets"
]

cv = CountVectorizer()
X_train = cv.fit_transform(train_docs)

print("Training vocabulary:")
print(cv.get_feature_names_out())

new_doc = ["hamster eats seeds"]
X_new = cv.transform(new_doc)

print("\nNew document vector:")
display(pd.DataFrame(
    X_new.toarray(),
    columns=cv.get_feature_names_out()
))

print("\nOOV tokens in new document:")
train_vocab = set(cv.get_feature_names_out())
print([t for t in simple_tokenize(new_doc[0]) if t not in train_vocab])

Training vocabulary:
['and' 'are' 'cat' 'dog' 'eats' 'fish' 'meat' 'pets']

New document vector:


,and,are,cat,dog,eats,fish,meat,pets
0,0,0,0,0,1,0,0,0



OOV tokens in new document:
['hamster', 'seeds']


### Một số chiến lược khi gặp OOV

1. Dùng token `<UNK>` trong các pipeline tự xây dựng.
2. Giới hạn vocabulary bằng `min_df`, `max_df`, `max_features`.
3. Dùng character n-grams để giảm độ nhạy với từ lạ / lỗi chính tả.
4. Với mô hình hiện đại: subword tokenization (BPE, WordPiece, SentencePiece).

### Character n-gram demo

In [17]:
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
X_char = char_vec.fit_transform([
    "classification",
    "classifier",
    "classify"
])

print("Shape:", X_char.shape)
print("Similarity:")
display(pd.DataFrame(
    cosine_similarity(X_char),
    index=["classification", "classifier", "classify"],
    columns=["classification", "classifier", "classify"]
).round(3))

Shape: (3, 54)
Similarity:


,classification,classifier,classify
classification,1.000,0.330,0.295
classifier,0.330,1.000,0.391
classify,0.295,0.391,1.000


## Demo 8. Lưu ý với tiếng Việt

Dấu cách trong tiếng Việt không phải lúc nào cũng tương ứng 1–1 với ranh giới từ.
Ví dụ: **"trí tuệ nhân tạo"** gồm nhiều âm tiết.

Trong notebook này ta tập trung vào **text representation**; với bài toán thực tế,
bước word segmentation/tokenization tiếng Việt nên được xử lý bằng tokenizer phù hợp trước khi vector hóa.

In [18]:
vi_docs = [
    "trí_tuệ_nhân_tạo xử_lý ngôn_ngữ",
    "học_máy phân_loại văn_bản",
    "xử_lý_ngôn_ngữ dùng dữ_liệu văn_bản"
]

vi_vec = CountVectorizer(token_pattern=r"(?u)\b\w[\w_]+\b")
X_vi = vi_vec.fit_transform(vi_docs)

pd.DataFrame(
    X_vi.toarray(),
    columns=vi_vec.get_feature_names_out(),
    index=["V1", "V2", "V3"]
)

,dùng,dữ_liệu,học_máy,ngôn_ngữ,phân_loại,trí_tuệ_nhân_tạo,văn_bản,xử_lý,xử_lý_ngôn_ngữ
V1,0,0,0,1,0,1,0,1,0
V2,0,0,1,0,1,0,1,0,0
V3,1,1,0,0,0,0,1,0,1


# Part B — Guided Practice

## Practice 1. Tự xây vocabulary và BoW

Corpus:

In [19]:
practice_docs = [
    "deep learning for text",
    "deep learning for images",
    "text mining and language processing"
]

practice_docs

['deep learning for text',
 'deep learning for images',
 'text mining and language processing']

**Nhiệm vụ**

1. Tokenize các document.
2. Tạo vocabulary đã sắp xếp alphabet.
3. Tạo count vector cho từng document.
4. Tạo DataFrame DTM.

In [20]:
# TODO: hoàn thiện
practice_tokens = [simple_tokenize(d) for d in practice_docs]
practice_vocab = sorted(set(token for doc in practice_tokens for token in doc))

def get_count_vector(tokens, vocab):
    counts = Counter(tokens)
    return [counts[word] for word in vocab]

practice_dtm = pd.DataFrame(
    [get_count_vector(tokens, practice_vocab) for tokens in practice_tokens],
    index=[f"D{i}" for i in range(1, len(practice_docs) + 1)],
    columns=practice_vocab
)

display(practice_dtm)

## Practice 2. `CountVectorizer`

Dùng `CountVectorizer` để:
- tạo DTM;
- lấy vocabulary;
- xác định 5 term có tổng tần suất cao nhất trong corpus.

In [21]:
# TODO
cv_practice = CountVectorizer()
X_practice = cv_practice.fit_transform(practice_docs)

# Vocabulary
print("Vocabulary:", cv_practice.get_feature_names_out())

# 5 term có tổng tần suất cao nhất
term_counts = np.asarray(X_practice.sum(axis=0)).ravel()
feature_names = cv_practice.get_feature_names_out()
top5_idx = term_counts.argsort()[::-1][:5]

print("\nTop 5 terms có tổng tần suất cao nhất:")
for idx in top5_idx:
    print(f"- {feature_names[idx]}: {term_counts[idx]}")

## Practice 3. Top TF–IDF terms

Với mỗi document, tìm **3 term có TF–IDF lớn nhất**.

In [22]:
# TODO
tfidf_practice = TfidfVectorizer()
Xp = tfidf_practice.fit_transform(practice_docs)
feature_names = tfidf_practice.get_feature_names_out()

for i, doc in enumerate(practice_docs, start=1):
    row = Xp[i - 1].toarray().ravel()
    top3_idx = row.argsort()[::-1][:3]
    top3_terms = [(feature_names[idx], round(row[idx], 4)) for idx in top3_idx if row[idx] > 0]
    print(f"Document {i}: '{doc}'")
    print("  Top 3 TF-IDF terms:", top3_terms)

## Practice 4. Mini search engine

Corpus:

In [23]:
search_docs = [
    "python is a popular programming language",
    "natural language processing analyzes human language",
    "machine learning learns patterns from data",
    "python is widely used for machine learning",
    "information retrieval ranks documents for a query",
]

query = "python machine learning"

**Nhiệm vụ**

1. Fit `TfidfVectorizer` trên `search_docs`.
2. Transform `query`.
3. Tính cosine similarity.
4. Trả về ranking document từ cao xuống thấp.

In [24]:
# TODO
vec_search = TfidfVectorizer()
X_search = vec_search.fit_transform(search_docs)
query_vec = vec_search.transform([query])

sims = cosine_similarity(query_vec, X_search).ravel()
ranked_indices = sims.argsort()[::-1]

print(f"Query: '{query}'\nRanking:")
for rank, idx in enumerate(ranked_indices, start=1):
    print(f"{rank}. [Score: {sims[idx]:.4f}] (Doc {idx+1}): {search_docs[idx]}")

## Practice 5. OOV

Fit vectorizer trên 3 câu đầu, sau đó transform câu mới:

`transformers understand context`

Trả lời:
- những token nào là OOV?
- vector kết quả có bao nhiêu phần tử khác 0?
- vì sao **không nên** `fit_transform()` lại trên test set?

In [25]:
oov_train = [
    "neural networks learn representations",
    "machine learning uses data",
    "text classification uses features"
]
oov_test = ["transformers understand context"]

# TODO
vec_oov = CountVectorizer()
vec_oov.fit(oov_train)
test_vec = vec_oov.transform(oov_test)

train_vocab = set(vec_oov.get_feature_names_out())
test_tokens = simple_tokenize(oov_test[0])
oov_tokens = [w for w in test_tokens if w not in train_vocab]

print("Vocabulary từ train set:", sorted(train_vocab))
print("Các token OOV trong test set:", oov_tokens)
print("Số phần tử khác 0 trong vector kết quả:", test_vec.nnz)
print("\nGiải thích:")
print("- Tất cả các từ trong câu test ('transformers', 'understand', 'context') đều là OOV.")
print("- Vector kết quả có 0 phần tử khác 0 (toàn bộ là 0).")
print("- Không nên fit_transform() lại trên test set vì gây data leakage và làm sai lệch không gian đặc trưng giữa train và test.")

# Part C — Exercises

## Exercise 1. BoW bằng tay

Cho corpus:

- D1: `data science uses data`
- D2: `machine learning uses data`
- D3: `science and machine learning`

Yêu cầu:

1. Xây vocabulary theo alphabet.
2. Viết count vector của D1, D2, D3.
3. Tạo Document–Term Matrix.
4. Term nào có document frequency lớn nhất?

In [26]:
ex1_docs = [
    "data science uses data",
    "machine learning uses data",
    "science and machine learning"
]

# TODO
# 1. Vocabulary sắp xếp alphabet
ex1_tokens = [simple_tokenize(d) for d in ex1_docs]
ex1_vocab = sorted(set(t for doc in ex1_tokens for t in doc))

# 2 & 3. Count vectors và Document-Term Matrix
ex1_vectors = [[Counter(doc)[w] for w in ex1_vocab] for doc in ex1_tokens]
ex1_dtm = pd.DataFrame(ex1_vectors, index=["D1", "D2", "D3"], columns=ex1_vocab)
display(ex1_dtm)

# 4. Term có document frequency lớn nhất
df_series = (ex1_dtm > 0).sum(axis=0)
max_df = df_series.max()
max_df_terms = df_series[df_series == max_df].index.tolist()
print("\nDocument frequency (df) của từng term:")
print(df_series)
print(f"\nTerm(s) có document frequency lớn nhất (df = {max_df}):", max_df_terms)

## Exercise 2. TF–IDF bằng tay

Dùng corpus Exercise 1.

Với công thức:

\[
idf(t)=\log(N/df(t))+1
\]

Hãy:

1. tính `df` của `data`, `science`, `machine`;
2. tính IDF của ba từ;
3. tính TF–IDF của ba từ trong từng document;
4. giải thích từ nào có tính phân biệt cao hơn.

In [27]:
# TODO
# N = 3 documents
N = 3
target_terms = ["data", "science", "machine"]

# 1. Tính df của 3 từ
df_dict = {t: sum(1 for doc in ex1_tokens if t in doc) for t in target_terms}

# 2. Tính IDF: idf(t) = log(N / df(t)) + 1
idf_dict = {t: math.log(N / df_dict[t]) + 1 for t in target_terms}

print("1. Document Frequency (df):", df_dict)
print("2. Inverse Document Frequency (IDF):", {k: round(v, 4) for k, v in idf_dict.items()})

# 3. Tính TF-IDF của 3 từ trong từng document
tfidf_manual = {}
for i, doc in enumerate(ex1_tokens, start=1):
    counts = Counter(doc)
    tfidf_manual[f"D{i}"] = {t: round(counts[t] * idf_dict[t], 4) for t in target_terms}

print("\n3. Bảng TF-IDF:")
tfidf_df = pd.DataFrame(tfidf_manual).T
display(tfidf_df)

print("\n4. Giải thích tính phân biệt:")
print("- Cả 3 từ đều xuất hiện trong 2/3 documents nên có cùng giá trị IDF ≈ 1.4055.")
print("- Trong D1, từ 'data' có TF = 2 (TF-IDF = 2.8109), phân biệt mạnh D1 so với các tài liệu khác.")

## Exercise 3. Document similarity

Dùng TF–IDF và cosine similarity để tìm cặp document giống nhau nhất
trong corpus sau:

In [28]:
ex3_docs = [
    "deep learning for computer vision",
    "neural networks for image recognition",
    "natural language processing for text",
    "text classification with machine learning",
    "computer vision and image processing"
]

# TODO
vec3 = TfidfVectorizer()
X3 = vec3.fit_transform(ex3_docs)
sim_matrix = cosine_similarity(X3)

max_sim = -1
best_pair = None
n = len(ex3_docs)
for i in range(n):
    for j in range(i + 1, n):
        if sim_matrix[i, j] > max_sim:
            max_sim = sim_matrix[i, j]
            best_pair = (i, j)

print(f"Cặp document giống nhau nhất: D{best_pair[0]+1} và D{best_pair[1]+1} (Cosine similarity = {max_sim:.4f})")
print(f"- D{best_pair[0]+1}: '{ex3_docs[best_pair[0]]}'")
print(f"- D{best_pair[1]+1}: '{ex3_docs[best_pair[1]]}'")

## Exercise 4. Search engine nhỏ

Viết hàm:

```python
rank_documents(query, documents, top_k=3)
```

trả về `top_k` document có cosine similarity lớn nhất dựa trên TF–IDF.

Yêu cầu output có:
- document id;
- text;
- similarity score.

In [29]:
def rank_documents(query, documents, top_k=3):
    # TODO
    vectorizer = TfidfVectorizer()
    X_docs = vectorizer.fit_transform(documents)
    query_vec = vectorizer.transform([query])
    
    similarities = cosine_similarity(query_vec, X_docs).ravel()
    top_indices = similarities.argsort()[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "doc_id": idx,
            "text": documents[idx],
            "similarity_score": round(float(similarities[idx]), 4)
        })
    return results

# Thử nghiệm hàm
rankings = rank_documents("deep learning vision", ex3_docs, top_k=3)
display(pd.DataFrame(rankings))

## Exercise 5. Sparse matrix

Tạo corpus giả gồm:
- 1,000 documents;
- vocabulary tối đa 5,000 từ;
- mỗi document dùng khoảng 30 từ.

So sánh:
- shape;
- `nnz`;
- density;
- ước lượng bộ nhớ dense;
- bộ nhớ sparse CSR thực tế.

In [30]:
# TODO
# Tạo corpus giả lập: 1,000 documents, vocab <= 5,000 words, mỗi doc khoảng 30 từ
np.random.seed(42)
n_docs = 1000
vocab_pool = [f"word_{i}" for i in range(5000)]

fake_docs = []
for _ in range(n_docs):
    doc_len = np.random.randint(25, 36)
    words = np.random.choice(vocab_pool, size=doc_len, replace=True)
    fake_docs.append(" ".join(words))

cv_sparse = CountVectorizer()
X_csr = cv_sparse.fit_transform(fake_docs)

# Các chỉ số so sánh
shape = X_csr.shape
nnz = X_csr.nnz
total_elements = shape[0] * shape[1]
density = nnz / total_elements

# Bộ nhớ ước tính dense (int64: 8 bytes/phần tử) và sparse CSR thực tế
dense_mem_bytes = total_elements * 8
sparse_mem_bytes = X_csr.data.nbytes + X_csr.indices.nbytes + X_csr.indptr.nbytes

print("=== SO SÁNH MA TRẬN DENSE VS SPARSE ===")
print(f"- Shape: {shape}")
print(f"- Số phần tử khác 0 (nnz): {nnz:,}")
print(f"- Density (độ đậm đặc): {density:.4%}")
print(f"- Ước lượng bộ nhớ Dense (int64): {dense_mem_bytes / (1024**2):.2f} MB")
print(f"- Bộ nhớ Sparse CSR thực tế: {sparse_mem_bytes / 1024:.2f} KB")
print(f"=> Sparse tiết kiệm khoảng {dense_mem_bytes / sparse_mem_bytes:.1f} lần bộ nhớ.")

## Exercise 6. OOV và data leakage

Giả sử:
- train: `["cat eats fish", "dog eats meat"]`
- test: `["hamster eats seeds"]`

Thực hiện hai cách:

**Cách A**
```python
vectorizer.fit(train)
vectorizer.transform(test)
```

**Cách B**
```python
vectorizer.fit(train + test)
```

So sánh vocabulary và giải thích tại sao cách B gây **data leakage**.

In [31]:
train = ["cat eats fish", "dog eats meat"]
test = ["hamster eats seeds"]

# TODO
# Cách A: Fit trên train, transform trên test
vec_a = CountVectorizer()
vec_a.fit(train)
vocab_a = vec_a.get_feature_names_out()
test_vec_a = vec_a.transform(test)

# Cách B: Fit trên cả train + test
vec_b = CountVectorizer()
vec_b.fit(train + test)
vocab_b = vec_b.get_feature_names_out()
test_vec_b = vec_b.transform(test)

print("Vocabulary Cách A (chỉ train):", list(vocab_a))
print("Vocabulary Cách B (train + test):", list(vocab_b))

print("\n=== GIẢI THÍCH DATA LEAKAGE ===")
print("- Cách B gây Data Leakage vì đưa thông tin từ test set ('hamster', 'seeds') vào từ điển của mô hình khi huấn luyện.")
print("- Trong thực tế, test set đại diện cho dữ liệu chưa biết trong tương lai. Nếu fit cả test set, đánh giá mô hình sẽ bị thiên lệch và không phản ánh đúng năng lực xử lý từ mới (OOV).")

## Exercise 7 — Bonus. Word n-gram vs Character n-gram

So sánh similarity của:

- `color`
- `colour`
- `colors`
- `colored`

với:
1. word-level TF–IDF;
2. character n-gram TF–IDF (`analyzer="char_wb"`, `ngram_range=(3,5)`).

Viết nhận xét: character n-gram giúp gì cho OOV / biến thể chính tả?

In [32]:
words = ["color", "colour", "colors", "colored"]

# TODO
# 1. Word-level TF-IDF
vec_word = TfidfVectorizer(analyzer="word")
X_word = vec_word.fit_transform(words)
sim_word = cosine_similarity(X_word)

# 2. Character n-gram TF-IDF
vec_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
X_char = vec_char.fit_transform(words)
sim_char = cosine_similarity(X_char)

print("=== 1. Similarity ma trận (Word-level TF-IDF) ===")
display(pd.DataFrame(sim_word, index=words, columns=words))

print("\n=== 2. Similarity ma trận (Character n-gram TF-IDF) ===")
display(pd.DataFrame(np.round(sim_char, 4), index=words, columns=words))

print("\n=== NHẬN XÉT ===")
print("- Word-level coi mỗi từ biến thể là một token độc lập hoàn toàn, độ tương đồng giữa các từ khác nhau bằng 0.")
print("- Character n-gram trích xuất các n-gram ký tự con, giúp mô hình bắt được độ tương đồng cao (> 0.5) giữa các từ có cùng gốc/biến thể chính tả.")
print("- Do đó, Character n-gram giúp xử lý hiệu quả vấn đề OOV và lỗi chính tả.")

# Discussion questions

1. BoW mất thông tin gì về ngôn ngữ?
2. Vì sao TF–IDF thường phù hợp hơn raw counts cho information retrieval?
3. Cosine similarity khác Euclidean distance ở điểm nào?
4. Khi nào sparse representation đặc biệt quan trọng?
5. OOV có thể ảnh hưởng mô hình downstream như thế nào?
6. Vì sao word embeddings / contextual embeddings được phát triển sau BoW/TF–IDF?

# Takeaways

- Text phải được ánh xạ sang vector trước khi phần lớn thuật toán ML có thể xử lý.
- BoW đơn giản, dễ giải thích nhưng bỏ qua trật tự và phần lớn ngữ nghĩa.
- Count vectors biểu diễn tần suất; TF–IDF giảm trọng số của từ quá phổ biến.
- DTM trong NLP thường rất sparse.
- Cosine similarity là phép đo cốt lõi cho so sánh vector văn bản.
- Vocabulary được học từ training data; từ ngoài vocabulary tạo ra vấn đề OOV.
- Các phương pháp embedding hiện đại giải quyết nhiều hạn chế của biểu diễn dựa trên vocabulary cố định.